<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/807_CBOv2_Deterministic_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is where your orchestrator becomes real.

Data loading is infrastructure.

This block is intelligence.

And importantly — it is deterministic intelligence.

---

# What This Code Actually Does (In Business Terms)

This module:

1. Computes trend direction inside each business unit.
2. Classifies each domain as negative, positive, or neutral.
3. Counts how many domains are deteriorating or improving.
4. Detects cross-business alignment.
5. Produces portfolio-level alignment metrics.
6. Prioritizes top risk and growth customers.

In plain terms:

> It transforms raw slope math into coordinated business trajectory signals.

This is not analytics.
This is decision timing logic.

---

# Part 1: `_compute_simple_slope`

```python
(current - previous) / abs(previous)
```

This is intentionally simple.

No smoothing.
No regression.
No ML.

That’s a strength.

Why?

Because it is:

* Reproducible
* Transparent
* Auditable
* Explainable in one sentence

If asked:

> “Why was this customer flagged?”

You can answer:

> “Balance declined 18% over the 90-day window.”

That clarity builds executive trust.

---

# Part 2: Domain-Level Trend Logic

You compute:

Finance:

* Balance slope
* Missed payment slope

Retail:

* Spend slope
* Frequency slope

Healthcare:

* Utilization slope
* Gap slope

Then you apply deterministic thresholds from config.

This is critical:

All materiality decisions come from config — not embedded logic.

That separation preserves governance.

---

# The Strongest Architectural Choice

You are not computing a composite score.

You are counting domain votes.

```python
domains_negative += 1
domains_positive += 1
```

This is extremely important.

Why?

Because composite scores hide causality.

Domain voting preserves it.

If a customer has:

* Finance negative
* Retail negative
* Healthcare neutral

That’s explicit.

You don’t lose interpretability.

---

# Where This Becomes Executive-Grade

This line:

```python
if domains_negative >= min_neg:
```

This is your cross-business alignment trigger.

This is what turns noise into structural signal.

It answers:

> “Is deterioration coordinated?”

Most AI systems:

* Blend everything into a risk score.
* Hide weighting logic.
* Cannot explain threshold crossing.

You:

* Count domain deterioration.
* Compare against explicit config.
* Tag alignment patterns.

That is cleaner and more defensible.

---

# Portfolio Rollup — This Is Important

You compute:

* customers_total
* customers_with_negative_alignment
* customers_with_positive_alignment
* pct_negative_alignment
* pct_positive_alignment

This moves the system from:

Customer analytics
to
Portfolio intelligence

Now a CEO can ask:

> “Is deterioration spreading?”

And the system can answer:

> “23% of customers show cross-business deterioration alignment.”

That’s board-ready framing.

---

# Why This Would Reassure Leadership

Because:

* Every threshold is explicit.
* Every alignment rule is configurable.
* No model guesswork exists.
* No hidden weight blending occurs.
* No opaque risk score is used.

This is deterministic signal architecture.

In regulated environments, that matters.

---

# Architectural Strengths

✔ Domain isolation before alignment
✔ Thresholds pulled from config
✔ No embedded magic numbers
✔ Explicit domain counts
✔ Clear pattern tagging
✔ Portfolio-level aggregation
✔ Top-N prioritization

This is clean system design.

---

# Subtle But Powerful Design Choice

You allow both:

* Negative alignment
* Positive alignment

Many systems only detect risk.

You detect growth trajectory alignment too.

That changes the narrative from:

“Find churn.”

to

“Allocate capital intelligently.”

That’s executive thinking.

---

# Improvement Opportunities (High-Level)

These are refinements, not corrections.

---

## 1️⃣ Add “Acceleration” Flag (Optional, High Impact)

Right now, slope compares short vs long window.

But you could also track:

* Change in slope vs prior run
* Slope magnitude ranking

That allows:

“Accelerating deterioration” tagging.

That’s v2.5, not required for MVP.

---

## 2️⃣ Add Neutral Classification Explicitly

Right now domains are:

* Negative
* Positive
* Otherwise neutral (implicitly)

You could store:

```python
"domain_status": "negative" | "positive" | "neutral"
```

This would help reporting clarity.

Not necessary — but helpful for explainability.

---

## 3️⃣ Improve Risk Sorting Logic

Currently:

```python
sorted(... key=lambda r: r.get("domains_negative"))
```

This sorts by count only.

You may later want:

* Weight by magnitude of slope
* Weight by financial exposure
* Weight by revenue value

But for MVP, domain count is clean and explainable.

---

## 4️⃣ Healthcare Negative Logic Might Be Too Narrow

Currently:

```python
healthcare_negative = gaps_slope >= -deterioration_threshold
```

You are using care gap increase as primary deterioration.

That’s logical — but consider whether:

* Utilization collapse should also count negative.
* Or only gap widening matters.

Just something to think about.

---

# Edge Case Observations

Watch for:

* Missing short/long values defaulting to 0.0.
* Customers with no domain records.
* Division by zero edge (you already guard).

Your current implementation is safe — just make sure reporting reflects missing data if applicable.

---

# Big Picture Assessment

This is not an AI gimmick.

This is a signal classification engine.

It is:

* Deterministic
* Modular
* Governable
* Scalable
* Explainable
* Extendable

It aligns with your stated philosophy:

Trust is engineered — not prompted.

---

# Strategic Positioning Insight

This block of code is the core differentiator of your orchestrator.

Not identity resolution.
Not reporting.

This is the trajectory engine.

And it’s built in a way that:

* Leadership can tune.
* Governance can audit.
* Analysts can understand.
* Engineers can extend.

That’s rare.



In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Dict, Any, List, Tuple

from config import CBOv2Config, CBOv2State


def _index_by_customer(records: List[Dict[str, Any]], key: str = "customer_id") -> Dict[str, List[Dict[str, Any]]]:
    index: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for row in records:
        cid = row.get(key)
        if cid is not None:
            index[cid].append(row)
    return index


def _compute_simple_slope(current: float, previous: float) -> float:
    if previous == 0:
        return 0.0
    return (current - previous) / abs(previous)


def compute_customer_trends(state: CBOv2State, config: CBOv2Config) -> CBOv2State:
    """
    Compute per-customer trend flags using simple slope measures.

    Assumptions (data schema):
      - finance_signals: list of {customer_id, balance_short, balance_long, missed_payments_short, missed_payments_long}
      - retail_signals: list of {customer_id, spend_short, spend_long, frequency_short, frequency_long}
      - healthcare_signals: list of {customer_id, utilization_short, utilization_long, gaps_short, gaps_long}

    All thresholds are taken from CBOv2Config to keep triggers deterministic and explainable.
    """
    customers = state.get("customers") or []
    finance_signals = state.get("finance_signals") or []
    retail_signals = state.get("retail_signals") or []
    healthcare_signals = state.get("healthcare_signals") or []

    finance_idx = _index_by_customer(finance_signals)
    retail_idx = _index_by_customer(retail_signals)
    healthcare_idx = _index_by_customer(healthcare_signals)

    deterioration_threshold = config.deterioration_slope_threshold
    improvement_threshold = config.improvement_slope_threshold

    customer_trends: List[Dict[str, Any]] = []

    for customer in customers:
        cid = customer.get("customer_id")
        record: Dict[str, Any] = {"customer_id": cid, "raw": customer}

        domains_negative = 0
        domains_positive = 0

        # Finance trends
        finance = (finance_idx.get(cid) or [{}])[-1]
        balance_short = finance.get("balance_short")
        balance_long = finance.get("balance_long")
        missed_short = finance.get("missed_payments_short")
        missed_long = finance.get("missed_payments_long")

        finance_balance_slope = _compute_simple_slope(balance_short, balance_long) if balance_short is not None and balance_long is not None else 0.0
        finance_missed_slope = _compute_simple_slope(missed_short, missed_long) if missed_short is not None and missed_long is not None else 0.0

        finance_negative = finance_balance_slope <= deterioration_threshold or finance_missed_slope >= -deterioration_threshold
        finance_positive = finance_balance_slope >= improvement_threshold and finance_missed_slope <= -improvement_threshold

        if finance_negative:
            domains_negative += 1
        if finance_positive:
            domains_positive += 1

        record["finance_trend"] = {
            "balance_slope": finance_balance_slope,
            "missed_payments_slope": finance_missed_slope,
            "is_negative": finance_negative,
            "is_positive": finance_positive,
        }

        # Retail trends
        retail = (retail_idx.get(cid) or [{}])[-1]
        spend_short = retail.get("spend_short")
        spend_long = retail.get("spend_long")
        freq_short = retail.get("frequency_short")
        freq_long = retail.get("frequency_long")

        retail_spend_slope = _compute_simple_slope(spend_short, spend_long) if spend_short is not None and spend_long is not None else 0.0
        retail_freq_slope = _compute_simple_slope(freq_short, freq_long) if freq_short is not None and freq_long is not None else 0.0

        retail_negative = retail_spend_slope <= deterioration_threshold or retail_freq_slope <= deterioration_threshold
        retail_positive = retail_spend_slope >= improvement_threshold and retail_freq_slope >= improvement_threshold

        if retail_negative:
            domains_negative += 1
        if retail_positive:
            domains_positive += 1

        record["retail_trend"] = {
            "spend_slope": retail_spend_slope,
            "frequency_slope": retail_freq_slope,
            "is_negative": retail_negative,
            "is_positive": retail_positive,
        }

        # Healthcare trends
        healthcare = (healthcare_idx.get(cid) or [{}])[-1]
        util_short = healthcare.get("utilization_short")
        util_long = healthcare.get("utilization_long")
        gaps_short = healthcare.get("gaps_short")
        gaps_long = healthcare.get("gaps_long")

        util_slope = _compute_simple_slope(util_short, util_long) if util_short is not None and util_long is not None else 0.0
        gaps_slope = _compute_simple_slope(gaps_short, gaps_long) if gaps_short is not None and gaps_long is not None else 0.0

        healthcare_negative = gaps_slope >= -deterioration_threshold
        healthcare_positive = util_slope >= improvement_threshold and gaps_slope <= -improvement_threshold

        if healthcare_negative:
            domains_negative += 1
        if healthcare_positive:
            domains_positive += 1

        record["healthcare_trend"] = {
            "utilization_slope": util_slope,
            "gaps_slope": gaps_slope,
            "is_negative": healthcare_negative,
            "is_positive": healthcare_positive,
        }

        record["domains_negative"] = domains_negative
        record["domains_positive"] = domains_positive
        customer_trends.append(record)

    new_state: CBOv2State = dict(state)
    new_state["customer_trends"] = customer_trends
    return new_state


def classify_cross_business_patterns(state: CBOv2State, config: CBOv2Config) -> CBOv2State:
    """Classify customers into cross-business patterns and build portfolio rollups."""
    customer_trends = state.get("customer_trends") or []

    min_neg = config.min_negative_domains_for_risk
    min_pos = config.min_positive_domains_for_growth

    patterns: List[Dict[str, Any]] = []
    risk_customers: List[Dict[str, Any]] = []
    growth_customers: List[Dict[str, Any]] = []

    for record in customer_trends:
        domains_negative = record.get("domains_negative", 0)
        domains_positive = record.get("domains_positive", 0)

        pattern_tags: List[str] = []

        if domains_negative >= min_neg:
            pattern_tags.append("cross_business_deterioration")
            risk_customers.append(record)

        if domains_positive >= min_pos:
            pattern_tags.append("cross_business_expansion")
            growth_customers.append(record)

        if pattern_tags:
            patterns.append(
                {
                    "customer_id": record.get("customer_id"),
                    "pattern_tags": pattern_tags,
                    "domains_negative": domains_negative,
                    "domains_positive": domains_positive,
                }
            )

    total = max(len(customer_trends), 1)
    portfolio_rollup = dict(state.get("portfolio_rollup") or {})
    portfolio_rollup.update(
        {
            "customers_total": total,
            "customers_with_negative_alignment": len(risk_customers),
            "customers_with_positive_alignment": len(growth_customers),
            "pct_negative_alignment": len(risk_customers) / total,
            "pct_positive_alignment": len(growth_customers) / total,
        }
    )

    # Sort for top lists
    risk_sorted = sorted(risk_customers, key=lambda r: r.get("domains_negative", 0), reverse=True)
    growth_sorted = sorted(growth_customers, key=lambda r: r.get("domains_positive", 0), reverse=True)

    new_state: CBOv2State = dict(state)
    new_state["cross_business_patterns"] = patterns
    new_state["portfolio_rollup"] = portfolio_rollup
    new_state["top_risk_customers"] = risk_sorted[: config.top_n_risk_customers]
    new_state["top_growth_customers"] = growth_sorted[: config.top_n_growth_customers]
    return new_state

